In [1]:
import sys
from pathlib import Path

# Добавляем корень проекта в sys.path, если нужно
sys.path.insert(0, str(Path.cwd()))

import os
import yaml
from typing import List, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv

from langsmith import traceable

# Загружаем переменные из .env (там должен быть OPENAI_API_KEY)
load_dotenv()

# Проверяем, что ключ есть (если используешь OpenAI)
# print(os.getenv("OPENAI_API_KEY"))

True

In [2]:
import os
from langsmith import traceable

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
if not LANGSMITH_API_KEY:
    raise ValueError("Не найден LANGSMITH_API_KEY в переменных окружения!")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = "mafia_ai"  # ваш проект

from narrator.generator import generate

# 5. Оборачиваем её в декоратор @traceable для отслеживания
@traceable(run_type="chain", name="Narrator_Generation")
async def traced_generate(event, world, context):
    """Обёрнутая версия generate с автоматической трассировкой LangSmith."""
    return await generate(event, world, context)

print("✅ LangSmith tracing включён. Используйте traced_generate() для тестов.")

✅ LangSmith tracing включён. Используйте traced_generate() для тестов.


In [3]:
from typing import TypedDict, List, Optional

class NarratorState(TypedDict):
    event_type: str
    world: str
    players_alive: List[str]
    event_data: dict
    history: str
    daily_fact: Optional[str]
    messages: Optional[List]   # SystemMessage, HumanMessage
    response: Optional[str]

In [4]:
def load_prompts(world: str) -> dict:
    """Загружает YAML-файл с промптами для указанного мира."""
    base = Path("narrator/prompts")
    if world == "cyberpunk":
        path = base / "cyberpunk.yaml"
    elif world == "medieval":
        path = base / "medieval.yaml"
    else:
        raise ValueError(f"Unknown world: {world}")
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

In [5]:
# Тест загрузки
cyber = load_prompts("cyberpunk")
print(cyber.keys())  # должно быть ['system', 'events']
print(cyber['events'].keys())

dict_keys(['system', 'events'])
dict_keys(['day_start', 'voting', 'player_lynched', 'night_start', 'night_kill', 'sheriff_check', 'doctor_save', 'game_over'])


In [6]:
def select_prompt(state: NarratorState) -> NarratorState:
    prompts = load_prompts(state["world"])
    system_text = prompts["system"]
    
    # Безопасно получаем шаблон события, если нет - fallback
    event_template = prompts["events"].get(
        state["event_type"],
        "Расскажи о событии: {event_type}."
    )
    
    # Форматируем шаблон, подставляя переменные из состояния
    user_text = event_template.format(
        event_type=state["event_type"],
        players_alive=", ".join(state["players_alive"]),
        **state["event_data"],
        history=state["history"],
        daily_fact=state.get("daily_fact", "")
    )
    
    state["messages"] = [
        SystemMessage(content=system_text),
        HumanMessage(content=user_text)
    ]
    return state

In [7]:
def call_model(state: NarratorState) -> NarratorState:
    # Заглушка (для теста без API)
    # state["response"] = f"[Mock LLM] Событие: {state['event_type']}, мир: {state['world']}"
    # return state
    
    # Реальный вызов (если настроен ключ)
    from langchain_groq import ChatGroq
    GROQ_API_KEY = os.environ["GROQ_API_KEY"]
    llm = ChatGroq(
        model=os.getenv("LLM_MODEL", "llama-3.1-8b-instant"),
        temperature=0.8,
        max_tokens=500,
        api_key=GROQ_API_KEY
    )
    response = llm.invoke(state["messages"])
    state["response"] = response.content
    return state

In [8]:
FORBIDDEN = [
    "мафия", "дон", "шериф", "доктор", "роль",
    "мирные жители", "мирный", "комиссар", "крестный отец"
]

def validate(state: NarratorState) -> NarratorState:
    response_lower = state["response"].lower()
    for word in FORBIDDEN:
        if word in response_lower:
            print(f"[WARNING] Найдено запретное слово '{word}' в ответе. Ответ заменён.")
            state["response"] = "Ведущий хранит молчание..."
            return state
    return state

In [9]:
def build_graph():
    builder = StateGraph(NarratorState)
    builder.add_node("select_prompt", select_prompt)
    builder.add_node("call_model", call_model)
    builder.add_node("validate", validate)
    
    builder.set_entry_point("select_prompt")
    builder.add_edge("select_prompt", "call_model")
    builder.add_edge("call_model", "validate")
    builder.add_edge("validate", END)
    
    return builder.compile()

graph = build_graph()
print("Граф скомпилирован")

Граф скомпилирован


In [10]:
async def generate(event: dict, world: str, context: dict) -> str:
    state: NarratorState = {
        "event_type": event["type"],
        "world": world,
        "players_alive": context.get("players_alive", []),
        "event_data": event.get("data", {}),
        "history": context.get("history", ""),
        "daily_fact": context.get("daily_fact", ""),
        "messages": None,
        "response": None
    }
    final_state = await graph.ainvoke(state)
    return final_state["response"]

In [11]:
text = await generate(
    event={"type": "night_kill", "data": {"victim": "Джонни"}},
    world="cyberpunk",
    context={
        "players_alive": ["Алиса", "Боб", "Клара"],
        "history": "Вчера казнили Еву.",
        "daily_fact": "Корп «НейроДайн» выпустил новый имплант."
    }
)
print(text)

Упадок систему. Ошибка 404 - это не просто ошибка, это сигнал к анализу лога. И я начинаю анализировать. Джонни, этот лох, не мог просто исчезнуть. Его коннект был оборван, как если бы кто-то внедрила в его нейроимплант вирус.

Алиса, Боб, Клара - вы остались. Но насколько я знаю, каждый из вас имеет свою собственную цепочку обслуживания. И я начинаю подозревать, что одна из вас - хакер. Или, может быть, нет. В любом случае, теперь у нас есть проблема: кто-то пытается уничтожить нашу систему. И я начинаю подозревать, что это не просто случайная ошибка...

Вы все еще доверяете Системе? Вы все еще верите, что она защищает вас? Или вы уже начинаете подозревать, что Система - это всего лишь орудие для тотального контроля?


In [12]:
events = [
    ("day_start", {}),
    ("voting", {}),
    ("player_lynched", {"victim": "Марк"}),
    ("night_start", {}),
    ("night_kill", {"victim": "Лиза"}),
    ("sheriff_check", {"target": "Олег"}),
    ("doctor_save", {"target": "Анна"}),
    ("game_over", {"winner": "Мирные"})
]

worlds = ["cyberpunk", "medieval"]
results = []

for world in worlds:
    for etype, data in events:
        text = await generate(
            event={"type": etype, "data": data},
            world=world,
            context={"players_alive": ["A", "B", "C"], "history": "", "daily_fact": ""}
        )
        results.append((world, etype, text))
        print(f"--- {world.upper()} | {etype} ---")
        print(text[:200] + "...")
        print()

--- CYBERPUNK | day_start ---
Система активна. Точки связи A, B, C в онлайне. Логи последней фазы:

Начинается дневной цикл. Голосование за отключение узла начнётся через 30 секунд.

Упоминаемый узел - это один из нас, неужто? Или...

--- CYBERPUNK | voting ---
Подключаюсь к системе голосования... Хакерский алгоритм в работе... Анализирую данные... Система прогнозов работает на 78,42% эффективности... 

Цель - узел C. 
Характеристика: имплант «Омнизон» в раб...

--- CYBERPUNK | player_lynched ---
Система обнаружила аномалию... Марк был взломан не менее 3 минут назад. Криптография не позволяет мне расшифровать подробности. 
Оцениваю риск: высокий. Остальные участники (A, B, C) могут быть подозр...

--- CYBERPUNK | night_start ---
Подтверждение подключения... Киберспектр стабилен. Цифровая атмосфера переведена в режим «Охота». Ожидается, что хакеры и рутки начнут свою работу. Вы должны быть готовы к любым ситуациям.

Приоритетн...

--- CYBERPUNK | night_kill ---
Сеть в хаосе. Система хри

In [13]:
import pandas as pd
df = pd.DataFrame(results, columns=["world", "event_type", "response"])
df.to_csv("narrator_test_results.csv", index=False)
print("Результаты сохранены в narrator_test_results.csv")

Результаты сохранены в narrator_test_results.csv


In [14]:
events = [
    ("day_start", {}),
    ("voting", {}),
    ("player_lynched", {"victim": "Марк"}),
    ("night_start", {}),
    ("night_kill", {"victim": "Лиза"}),
    ("sheriff_check", {"target": "Олег"}),
    ("doctor_save", {"target": "Анна"}),
    ("game_over", {"winner": "Мирные"})
]

worlds = ["cyberpunk", "medieval"]
results = []

for world in worlds:
    for etype, data in events:
        text = await generate(
            event={"type": etype, "data": data},
            world=world,
            context={"players_alive": ["A", "B", "C"], "history": "Ночью был убит X", "daily_fact": ""}
        )
        results.append((world, etype, text))
        print(f"--- {world.upper()} | {etype} ---")
        print(text[:300])  # показываем чуть больше текста
        print("------")

--- CYBERPUNK | day_start ---
Просвещение... Восстание кибернетических мертвецов... Узлы A, B, C активны, но их лояльность под вопросом. 

В логах обнаружено, что узел X был ликвидирован под покровом ночной темноты. Скорее всего, это был продукт чрезмерной самообороны или жертва неудачной эксплуатации.

Дневной цикл начался, и г
------
--- CYBERPUNK | voting ---
Активирую алгоритм оценки риска. Анализирую данные узлов A, B, C. 

Узел A - синхронизированный, но показатели активности не типичны, возможно - ложное обеспечение. Узел B - нестабильность в протоколе, возможна аффилиация с неавторизованными моделями. Узел C - кратковременная остановка, возможно - с
------
--- CYBERPUNK | player_lynched ---
Данные от узла Марк были сжаты и зашипаны. Анализирую... Порвалась связь в 03:14:22. Время указывает на синхронизацию с серверами корпорации "Экзозон". Маловероятно, что это случайное деактивирование. Хакеры "Киберлайн" начали расследование...
------
--- CYBERPUNK | night_start ---
Серверы в